In [50]:
%pip install --upgrade nltk pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


# SchemeSathi Notebook 
This notebook is your project implementation for SchemeSathi.

What this notebook covers:
1. Install and import libraries
2. Load and clean dataset
3. Build eligibility-based recommendation engine (Phase 1 and 2)
4. Test with sample user profiles
5. Add optional NLP search and hybrid ranking

In [51]:
#importing necessary libraries
import pandas as pd
import numpy as np
import nltk
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import wordnet
from sklearn.feature_extraction.text import CountVectorizer
from nltk import pos_tag
from sklearn.metrics import pairwise_distances
from nltk import word_tokenize

#downloading necessary nltk resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_t

True

## Data Loading
In this section, we load the CSV file and quickly inspect the data before building the model.

In [74]:
# Load the main schemes dataset from local folder
dataset_path = "C:/Users/ASUS/OneDrive/Documents/GITHUB/SchemeSathi/Dataset/SchemeSathi-main-data.csv"
df = pd.read_csv(dataset_path)
print(f"Loaded rows: {len(df)}")

Loaded rows: 3400


In [53]:
df.head()

,scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,tags
0,"""Immediate Relief Assistance"" under ""Welfare a...",ira-wrflsncs,"The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",The applicant should be the family (legal heir...,Step 1: The interested applicant should visit ...,Photograph of the Family (Legal Heir) of the M...,State,"Agriculture,Rural & Environment, Social welfar...","Missing, Fisherman, Relief, Financial Assistan..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,"Short Term Training Programme-SFURTI Program, ...","Financial Assistance : Limit of funding ₹ 4,00...",The institution should be AICTE approved.,Registration of New Institute: Step 01: Visit ...,Feedback Form Copy of Proceedings Completion R...,Central,Education & Learning,"Trainings, Financial Assistance, AICTE"
2,Burial and Ex-gratia Payment Scheme in Case of...,baepsicodouldwact,"Launched in 2014, the "" Burial and Ex-gratia P...","Funeral Assistance: ₹3,000 payable in case of ...",The deceased construction worker should have b...,Step 1: The interested applicant should visit ...,Aadhaar Card of the applicant (nominee/Legal h...,State,Social welfare & Empowerment,"Building Worker, Construction Workers, Unregis..."
3,Consortia & Tender Marketing Scheme,ctms,Promotion of the product of Micro and Small En...,Enlistment of the Unit for participating in Go...,Micro & Small Enterprises registered with NSIC...,"Step 01: The application form, in the prescrib...",A passport size photograph of each of the Prop...,Central,Business & Entrepreneurship,"Goods And Services Marketing, Turnkey Projects..."
4,Garuda Scheme for Funeral Expense,gsfe,Andhra Pradesh Brahmin Welfare Corporation (AB...,"Financial Assistance of ₹10,000/- for funeral ...",The applicant should be a close relative of th...,Registration and apply Step 01: Applicants hav...,Passport-size Photograph of the Applicant Aadh...,State,Social welfare & Empowerment,"Social Welfare, Financial Assistance, Deceased..."


In [54]:
df.head(20)

,scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,tags
0,"""Immediate Relief Assistance"" under ""Welfare a...",ira-wrflsncs,"The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",The applicant should be the family (legal heir...,Step 1: The interested applicant should visit ...,Photograph of the Family (Legal Heir) of the M...,State,"Agriculture,Rural & Environment, Social welfar...","Missing, Fisherman, Relief, Financial Assistan..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,"Short Term Training Programme-SFURTI Program, ...","Financial Assistance : Limit of funding ₹ 4,00...",The institution should be AICTE approved.,Registration of New Institute: Step 01: Visit ...,Feedback Form Copy of Proceedings Completion R...,Central,Education & Learning,"Trainings, Financial Assistance, AICTE"
2,Burial and Ex-gratia Payment Scheme in Case of...,baepsicodouldwact,"Launched in 2014, the "" Burial and Ex-gratia P...","Funeral Assistance: ₹3,000 payable in case of ...",The deceased construction worker should have b...,Step 1: The interested applicant should visit ...,Aadhaar Card of the applicant (nominee/Legal h...,State,Social welfare & Empowerment,"Building Worker, Construction Workers, Unregis..."
3,Consortia & Tender Marketing Scheme,ctms,Promotion of the product of Micro and Small En...,Enlistment of the Unit for participating in Go...,Micro & Small Enterprises registered with NSIC...,"Step 01: The application form, in the prescrib...",A passport size photograph of each of the Prop...,Central,Business & Entrepreneurship,"Goods And Services Marketing, Turnkey Projects..."
4,Garuda Scheme for Funeral Expense,gsfe,Andhra Pradesh Brahmin Welfare Corporation (AB...,"Financial Assistance of ₹10,000/- for funeral ...",The applicant should be a close relative of th...,Registration and apply Step 01: Applicants hav...,Passport-size Photograph of the Applicant Aadh...,State,Social welfare & Empowerment,"Social Welfare, Financial Assistance, Deceased..."
5,Incentive For The Intra Caste Marriage within ...,ifticmwstc,"The scheme ""Incentive for Intra Caste Marriage...","An incentive of ₹2,00,000/- is provided to the...",The couple should be residing in Karnataka. Bo...,Step 1: Visit the official website of the Trib...,Aadhaar Card Bank Account Details Marriage Pro...,State,Social welfare & Empowerment,"Intra Caste, Marriage, Schedule Tribe, Incenti..."
6,Incentive Scheme for MSMEs in Powerloom Sector...,ismpsscis,The scheme “State Capital Investment Subsidy” ...,20% of Fixed Capital Investment on Plant and m...,The Scheme shall be generally applicable to al...,"A micro, small or medium enterprise in Powerlo...",A copy of the Memorandum of Association and Ar...,State,Business & Entrepreneurship,"Powerloom, Incentives, State Capital Investmen..."
7,Indira Mahila Shakti Udyam Protsahan Yojana,imspesy,The Indira Mahila Shakti Udyam Protsahan Yojan...,"Loan Limit Loan amount of ₹ 50,00,000/- to ind...",The applicant's age should be 18 years or more...,Registration Step-1: Applicant have to visit t...,Aadhar card. Address proof. Bank account state...,State,Women and Child,"Women, Empowerment, Development, Motivaction"
8,Mukhyamantri Shramik Aujaar Sahayata Yojana,msasy,The Labor Department of Chhattisgarh launched ...,Tool kits can given free of cost to the regist...,The Applicant must be a native of Chhattisgarh...,Registration of a Construction Worker Under C...,Bank passbook photocopy. Self-declaration cert...,State,Social welfare & Empowerment,"Financial Assistance, Construction Workers, La..."
9,Nirman Shramik Jeevan va Bhavishya Suraksha Yo...,nsjbsy,"Launched in 2016, the scheme ""Nirman Shramik J...",Pradhan Mantri Suraksha Bima Yojana (PMSBY) 10...,1. The applicant must be registered as a bene...,REGISTRATION Step 1: Visit the Official Portal...,Copy of Beneficiary Registration Identity Card...,State,Health & Wellness,"Medical, Insura

In [55]:
df.shape[0]

3400

In [56]:
df.shape[1]

10

In [57]:
df.isnull().sum()

scheme_name        0
slug               0
details            0
benefits           0
eligibility        0
application        2
documents         11
level              0
schemeCategory     0
tags              29
dtype: int64

Handling Missing Values

In [75]:
# Fill missing values using previous row value (quick cleaning step)
df.ffill(axis=0, inplace=True)

,scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,tags
0,"""Immediate Relief Assistance"" under ""Welfare a...",ira-wrflsncs,"The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",The applicant should be the family (legal heir...,Step 1: The interested applicant should visit ...,Photograph of the Family (Legal Heir) of the M...,State,"Agriculture,Rural & Environment, Social welfar...","Missing, Fisherman, Relief, Financial Assistan..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,"Short Term Training Programme-SFURTI Program, ...","Financial Assistance : Limit of funding ₹ 4,00...",The institution should be AICTE approved.,Registration of New Institute: Step 01: Visit ...,Feedback Form Copy of Proceedings Completion R...,Central,Education & Learning,"Trainings, Financial Assistance, AICTE"
2,Burial and Ex-gratia Payment Scheme in Case of...,baepsicodouldwact,"Launched in 2014, the "" Burial and Ex-gratia P...","Funeral Assistance: ₹3,000 payable in case of ...",The deceased construction worker should have b...,Step 1: The interested applicant should visit ...,Aadhaar Card of the applicant (nominee/Legal h...,State,Social welfare & Empowerment,"Building Worker, Construction Workers, Unregis..."
3,Consortia & Tender Marketing Scheme,ctms,Promotion of the product of Micro and Small En...,Enlistment of the Unit for participating in Go...,Micro & Small Enterprises registered with NSIC...,"Step 01: The application form, in the prescrib...",A passport size photograph of each of the Prop...,Central,Business & Entrepreneurship,"Goods And Services Marketing, Turnkey Projects..."
4,Garuda Scheme for Funeral Expense,gsfe,Andhra Pradesh Brahmin Welfare Corporation (AB...,"Financial Assistance of ₹10,000/- for funeral ...",The applicant should be a close relative of th...,Registration and apply Step 01: Applicants hav...,Passport-size Photograph of the Applicant Aadh...,State,Social welfare & Empowerment,"Social Welfare, Financial Assistance, Deceased..."
...,...,...,...,...,...,...,...,...,...,...
3395,“Ishan Uday” Special Scholarship Scheme For No...,iu-sss-ner,A scholarship scheme by the Ministry of Educat...,"1) UGC will award 10,000 (ten thousand) fresh...",The applicant must be a domicile of NER. The p...,New Registration : Step 1: Visit the Registrat...,Domicile certificate to be issued by the compe...,Central,Education & Learning,"Scholarship, Financial Assistance"
3396,“Medical Camps (Allopathy and Indian systems o...,mcacism-giapsca,The sub-scheme “Medical Camps (Allopathy and I...,Free Health Checkup and Free Medicines. Awaren...,The applicant should be 60 years or above in a...,Step 1: The interested applicant should visit ...,Aadhaar Card. Birth Certificate / Age Proof. R...,State,Health & Wellness,"Medicine, Health, Checkup, Senior Citizen, Doctor"
3397,“RAKSHAK PURASKARS’’ Award,rpa,The Directorate of Women and Child Development...,The recipients of the awarded are given a scro...,The applicant should have shown exceptional co...,Step 1: In the prescribed format of the applic...,Passport Size Photograph (Duly Attested by a G...,State,"Social welfare & Empowerment, Public Safety,La...","Award, Crime, Women, Girl, Violence, Harrassment"
3398,"“Training in Coir” Component of the ""Developme...",ticcotdocs,The scheme “Training in Coir” Component of the...,"Stipend of ₹1,500 per month per trainee during...",The applicant should be a native of Puducherry...,Step 1: The interested applicant should take p...,Residential Certificate (for verifying native ...,State,Skills & Employment,"Skill, Training, Coir, Rural, Poor, Employment"


In [59]:
df.head(20)

,scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,tags
0,"""Immediate Relief Assistance"" under ""Welfare a...",ira-wrflsncs,"The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",The applicant should be the family (legal heir...,Step 1: The interested applicant should visit ...,Photograph of the Family (Legal Heir) of the M...,State,"Agriculture,Rural & Environment, Social welfar...","Missing, Fisherman, Relief, Financial Assistan..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,"Short Term Training Programme-SFURTI Program, ...","Financial Assistance : Limit of funding ₹ 4,00...",The institution should be AICTE approved.,Registration of New Institute: Step 01: Visit ...,Feedback Form Copy of Proceedings Completion R...,Central,Education & Learning,"Trainings, Financial Assistance, AICTE"
2,Burial and Ex-gratia Payment Scheme in Case of...,baepsicodouldwact,"Launched in 2014, the "" Burial and Ex-gratia P...","Funeral Assistance: ₹3,000 payable in case of ...",The deceased construction worker should have b...,Step 1: The interested applicant should visit ...,Aadhaar Card of the applicant (nominee/Legal h...,State,Social welfare & Empowerment,"Building Worker, Construction Workers, Unregis..."
3,Consortia & Tender Marketing Scheme,ctms,Promotion of the product of Micro and Small En...,Enlistment of the Unit for participating in Go...,Micro & Small Enterprises registered with NSIC...,"Step 01: The application form, in the prescrib...",A passport size photograph of each of the Prop...,Central,Business & Entrepreneurship,"Goods And Services Marketing, Turnkey Projects..."
4,Garuda Scheme for Funeral Expense,gsfe,Andhra Pradesh Brahmin Welfare Corporation (AB...,"Financial Assistance of ₹10,000/- for funeral ...",The applicant should be a close relative of th...,Registration and apply Step 01: Applicants hav...,Passport-size Photograph of the Applicant Aadh...,State,Social welfare & Empowerment,"Social Welfare, Financial Assistance, Deceased..."
5,Incentive For The Intra Caste Marriage within ...,ifticmwstc,"The scheme ""Incentive for Intra Caste Marriage...","An incentive of ₹2,00,000/- is provided to the...",The couple should be residing in Karnataka. Bo...,Step 1: Visit the official website of the Trib...,Aadhaar Card Bank Account Details Marriage Pro...,State,Social welfare & Empowerment,"Intra Caste, Marriage, Schedule Tribe, Incenti..."
6,Incentive Scheme for MSMEs in Powerloom Sector...,ismpsscis,The scheme “State Capital Investment Subsidy” ...,20% of Fixed Capital Investment on Plant and m...,The Scheme shall be generally applicable to al...,"A micro, small or medium enterprise in Powerlo...",A copy of the Memorandum of Association and Ar...,State,Business & Entrepreneurship,"Powerloom, Incentives, State Capital Investmen..."
7,Indira Mahila Shakti Udyam Protsahan Yojana,imspesy,The Indira Mahila Shakti Udyam Protsahan Yojan...,"Loan Limit Loan amount of ₹ 50,00,000/- to ind...",The applicant's age should be 18 years or more...,Registration Step-1: Applicant have to visit t...,Aadhar card. Address proof. Bank account state...,State,Women and Child,"Women, Empowerment, Development, Motivaction"
8,Mukhyamantri Shramik Aujaar Sahayata Yojana,msasy,The Labor Department of Chhattisgarh launched ...,Tool kits can given free of cost to the regist...,The Applicant must be a native of Chhattisgarh...,Registration of a Construction Worker Under C...,Bank passbook photocopy. Self-declaration cert...,State,Social welfare & Empowerment,"Financial Assistance, Construction Workers, La..."
9,Nirman Shramik Jeevan va Bhavishya Suraksha Yo...,nsjbsy,"Launched in 2016, the scheme ""Nirman Shramik J...",Pradhan Mantri Suraksha Bima Yojana (PMSBY) 10...,1. The applicant must be registered as a bene...,REGISTRATION Step 1: Visit the Official Portal...,Copy of Beneficiary Registration Identity Card...,State,Health & Wellness,"Medical, Insura

In [60]:
df.isnull().sum()

scheme_name       0
slug              0
details           0
benefits          0
eligibility       0
application       0
documents         0
level             0
schemeCategory    0
tags              0
dtype: int64

In [76]:
# Remove extra unnamed column if it exists
df.drop(columns=["Unnamed: 9"], errors="ignore", inplace=True)
df.head()

,scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,tags
0,"""Immediate Relief Assistance"" under ""Welfare a...",ira-wrflsncs,"The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",The applicant should be the family (legal heir...,Step 1: The interested applicant should visit ...,Photograph of the Family (Legal Heir) of the M...,State,"Agriculture,Rural & Environment, Social welfar...","Missing, Fisherman, Relief, Financial Assistan..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,"Short Term Training Programme-SFURTI Program, ...","Financial Assistance : Limit of funding ₹ 4,00...",The institution should be AICTE approved.,Registration of New Institute: Step 01: Visit ...,Feedback Form Copy of Proceedings Completion R...,Central,Education & Learning,"Trainings, Financial Assistance, AICTE"
2,Burial and Ex-gratia Payment Scheme in Case of...,baepsicodouldwact,"Launched in 2014, the "" Burial and Ex-gratia P...","Funeral Assistance: ₹3,000 payable in case of ...",The deceased construction worker should have b...,Step 1: The interested applicant should visit ...,Aadhaar Card of the applicant (nominee/Legal h...,State,Social welfare & Empowerment,"Building Worker, Construction Workers, Unregis..."
3,Consortia & Tender Marketing Scheme,ctms,Promotion of the product of Micro and Small En...,Enlistment of the Unit for participating in Go...,Micro & Small Enterprises registered with NSIC...,"Step 01: The application form, in the prescrib...",A passport size photograph of each of the Prop...,Central,Business & Entrepreneurship,"Goods And Services Marketing, Turnkey Projects..."
4,Garuda Scheme for Funeral Expense,gsfe,Andhra Pradesh Brahmin Welfare Corporation (AB...,"Financial Assistance of ₹10,000/- for funeral ...",The applicant should be a close relative of th...,Registration and apply Step 01: Applicants hav...,Passport-size Photograph of the Applicant Aadh...,State,Social welfare & Empowerment,"Social Welfare, Financial Assistance, Deceased..."


In [62]:
df1=df.head(10)
df1

,scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,tags
0,"""Immediate Relief Assistance"" under ""Welfare a...",ira-wrflsncs,"The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",The applicant should be the family (legal heir...,Step 1: The interested applicant should visit ...,Photograph of the Family (Legal Heir) of the M...,State,"Agriculture,Rural & Environment, Social welfar...","Missing, Fisherman, Relief, Financial Assistan..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,"Short Term Training Programme-SFURTI Program, ...","Financial Assistance : Limit of funding ₹ 4,00...",The institution should be AICTE approved.,Registration of New Institute: Step 01: Visit ...,Feedback Form Copy of Proceedings Completion R...,Central,Education & Learning,"Trainings, Financial Assistance, AICTE"
2,Burial and Ex-gratia Payment Scheme in Case of...,baepsicodouldwact,"Launched in 2014, the "" Burial and Ex-gratia P...","Funeral Assistance: ₹3,000 payable in case of ...",The deceased construction worker should have b...,Step 1: The interested applicant should visit ...,Aadhaar Card of the applicant (nominee/Legal h...,State,Social welfare & Empowerment,"Building Worker, Construction Workers, Unregis..."
3,Consortia & Tender Marketing Scheme,ctms,Promotion of the product of Micro and Small En...,Enlistment of the Unit for participating in Go...,Micro & Small Enterprises registered with NSIC...,"Step 01: The application form, in the prescrib...",A passport size photograph of each of the Prop...,Central,Business & Entrepreneurship,"Goods And Services Marketing, Turnkey Projects..."
4,Garuda Scheme for Funeral Expense,gsfe,Andhra Pradesh Brahmin Welfare Corporation (AB...,"Financial Assistance of ₹10,000/- for funeral ...",The applicant should be a close relative of th...,Registration and apply Step 01: Applicants hav...,Passport-size Photograph of the Applicant Aadh...,State,Social welfare & Empowerment,"Social Welfare, Financial Assistance, Deceased..."
5,Incentive For The Intra Caste Marriage within ...,ifticmwstc,"The scheme ""Incentive for Intra Caste Marriage...","An incentive of ₹2,00,000/- is provided to the...",The couple should be residing in Karnataka. Bo...,Step 1: Visit the official website of the Trib...,Aadhaar Card Bank Account Details Marriage Pro...,State,Social welfare & Empowerment,"Intra Caste, Marriage, Schedule Tribe, Incenti..."
6,Incentive Scheme for MSMEs in Powerloom Sector...,ismpsscis,The scheme “State Capital Investment Subsidy” ...,20% of Fixed Capital Investment on Plant and m...,The Scheme shall be generally applicable to al...,"A micro, small or medium enterprise in Powerlo...",A copy of the Memorandum of Association and Ar...,State,Business & Entrepreneurship,"Powerloom, Incentives, State Capital Investmen..."
7,Indira Mahila Shakti Udyam Protsahan Yojana,imspesy,The Indira Mahila Shakti Udyam Protsahan Yojan...,"Loan Limit Loan amount of ₹ 50,00,000/- to ind...",The applicant's age should be 18 years or more...,Registration Step-1: Applicant have to visit t...,Aadhar card. Address proof. Bank account state...,State,Women and Child,"Women, Empowerment, Development, Motivaction"
8,Mukhyamantri Shramik Aujaar Sahayata Yojana,msasy,The Labor Department of Chhattisgarh launched ...,Tool kits can given free of cost to the regist...,The Applicant must be a native of Chhattisgarh...,Registration of a Construction Worker Under C...,Bank passbook photocopy. Self-declaration cert...,State,Social welfare & Empowerment,"Financial Assistance, Construction Workers, La..."
9,Nirman Shramik Jeevan va Bhavishya Suraksha Yo...,nsjbsy,"Launched in 2016, the scheme ""Nirman Shramik J...",Pradhan Mantri Suraksha Bima Yojana (PMSBY) 10...,1. The applicant must be registered as a bene...,REGISTRATION Step 1: Visit the Official Portal...,Copy of Beneficiary Registration Identity Card...,State,Health & Wellness,"Medical, Insura

In [63]:
def step1(x):
    for i in x:
        a = str(i).lower()
        p = re.sub(r'[^a-z0-9]', ' ', a)
        print(p)

In [64]:
step1(df1["scheme_name"])

 immediate relief assistance  under  welfare and relief for fishermen during lean seasons and natural calamities scheme 
aicte short term training programme sfurti scheme
burial and ex gratia payment scheme in case of death of unregistered laborer during work at construction site
consortia   tender marketing scheme
garuda scheme for funeral expense
incentive for the intra caste marriage within scheduled tribe community
incentive scheme for msmes in powerloom sector  state capital investment subsidy
indira mahila shakti udyam protsahan yojana
mukhyamantri shramik aujaar sahayata yojana
nirman shramik jeevan va bhavishya suraksha yojana


In [65]:
nltk.download('punkt_tab')
s= 'aicte short term training programme sfurti scheme'
words = word_tokenize(s)
print(words)

['aicte', 'short', 'term', 'training', 'programme', 'sfurti', 'scheme']


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Phase 1 and 2 Core Logic
Phase 1: Convert free-text eligibility rules into structured columns.
Phase 2: Score each scheme against a user profile and return best matches with reasons.

In [66]:
# Phase 1 + 2: Data structuring + eligibility matching engine
import os
import math
import re

# ---------- Phase 1: Prepare and structure dataset ----------
df_phase = df.copy()
df_phase.columns = [c.strip().lower().replace(" ", "_") for c in df_phase.columns]


def _pick_col(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    return None


scheme_col = _pick_col(df_phase, ["scheme_name", "scheme", "name", "title"])
eligibility_col = _pick_col(df_phase, ["eligibility", "eligibility_criteria", "eligible_beneficiaries"])
benefits_col = _pick_col(df_phase, ["benefits", "benefit", "description", "details"])
documents_col = _pick_col(df_phase, ["documents", "required_documents", "document_required"])
application_col = _pick_col(df_phase, ["application", "how_to_apply", "application_link", "apply_link", "website"])

if scheme_col is None or eligibility_col is None:
    raise ValueError("Required columns not found. Need at least scheme_name and eligibility-type columns.")

states_master = [
    "andhra pradesh", "arunachal pradesh", "assam", "bihar", "chhattisgarh", "goa", "gujarat", "haryana",
    "himachal pradesh", "jharkhand", "karnataka", "kerala", "madhya pradesh", "maharashtra", "manipur",
    "meghalaya", "mizoram", "nagaland", "odisha", "punjab", "rajasthan", "sikkim", "tamil nadu", "telangana",
    "tripura", "uttar pradesh", "uttarakhand", "west bengal", "delhi", "jammu and kashmir", "ladakh",
    "andaman and nicobar", "chandigarh", "dadra and nagar haveli and daman and diu", "lakshadweep", "puducherry"
]

category_keywords = ["general", "sc", "st", "obc", "minority", "ews"]
occupation_keywords = ["student", "farmer", "labour", "worker", "woman", "women", "entrepreneur", "senior citizen", "pregnant", "widow", "disabled"]


def parse_age_range(text):
    t = str(text).lower()

    between = re.search(r"(\d{1,2})\s*(?:to|-|–)\s*(\d{1,2})\s*(?:years?|yrs?)", t)
    if between:
        a, b = int(between.group(1)), int(between.group(2))
        return float(min(a, b)), float(max(a, b)), "between"

    min_match = re.search(r"(?:above|greater than|minimum|min|age\s*>=?)\s*(\d{1,2})", t)
    max_match = re.search(r"(?:below|less than|maximum|max|age\s*<=?|upto|up to)\s*(\d{1,2})", t)

    if min_match and max_match:
        min_age = float(min_match.group(1))
        max_age = float(max_match.group(1))
        if min_age > max_age:
            min_age, max_age = max_age, min_age
        return min_age, max_age, "between"

    if min_match:
        return float(min_match.group(1)), 120.0, "min_only"

    if max_match:
        return 0.0, float(max_match.group(1)), "max_only"

    return 0.0, 120.0, "none"


def parse_income_limit(text):
    t = str(text).lower().replace(",", "")

    if not any(k in t for k in ["income", "annual", "year", "monthly", "rs", "rupee", "rupaye", "₹"]):
        return float("inf")

    nums = [float(x) for x in re.findall(r"\b\d{4,8}\b", t)]
    if not nums:
        return float("inf")

    return max(nums)


def parse_state_list(text):
    t = str(text).lower()
    if "all india" in t or "all states" in t or "pan india" in t or "nationwide" in t:
        return ["all"]
    found = [s for s in states_master if s in t]
    return sorted(set(found))


def parse_keyword_list(text, keywords):
    t = str(text).lower()
    found = [k for k in keywords if k in t]
    return sorted(set(found))


df_phase["eligibility_raw"] = df_phase[eligibility_col].fillna("").astype(str)
df_phase[["min_age", "max_age", "age_rule_type"]] = df_phase["eligibility_raw"].apply(lambda x: pd.Series(parse_age_range(x)))
df_phase["max_income"] = df_phase["eligibility_raw"].apply(parse_income_limit)
df_phase["eligible_states"] = df_phase["eligibility_raw"].apply(parse_state_list)
df_phase["eligible_categories"] = df_phase["eligibility_raw"].apply(lambda x: parse_keyword_list(x, category_keywords))
df_phase["occupation_tags"] = df_phase["eligibility_raw"].apply(lambda x: parse_keyword_list(x, occupation_keywords))

# Optional export for teammates
out_path = "C:/Users/ASUS/OneDrive/Documents/GITHUB/SchemeSathi/Dataset/schemes_processed.csv"
df_phase.to_csv(out_path, index=False)
print(f"Processed dataset saved: {out_path}")


# ---------- Phase 2: Matching + ranking engine ----------
def get_eligible_schemes(user_profile, data=df_phase, top_k=10, threshold=55):
    """
    user_profile keys:
    age (int), state (str), income (float), category (str), occupation (str), special_status (list/str)
    """

    age = int(user_profile.get("age", 0) or 0)
    state = str(user_profile.get("state", "")).strip().lower()
    income = float(user_profile.get("income", 0) or 0)
    category = str(user_profile.get("category", "")).strip().lower()
    occupation = str(user_profile.get("occupation", "")).strip().lower()

    results = []

    for _, row in data.iterrows():
        score = 0
        matched_rules = []
        failed_rules = []

        # Age scoring with human-friendly explanation labels.
        min_age = float(row.get("min_age", 0) if pd.notna(row.get("min_age", 0)) else 0)
        max_age = float(row.get("max_age", 120) if pd.notna(row.get("max_age", 120)) else 120)
        age_rule_type = str(row.get("age_rule_type", "none"))

        age_ok = min_age <= age <= max_age
        if age_rule_type == "none":
            score += 8
            matched_rules.append("No specific age criteria mentioned")
        elif age_rule_type == "min_only":
            if age_ok:
                score += 20
                matched_rules.append(f"Age {int(min_age)}+")
            else:
                failed_rules.append(f"Age must be at least {int(min_age)}")
        elif age_rule_type == "max_only":
            if age_ok:
                score += 20
                matched_rules.append(f"Age up to {int(max_age)}")
            else:
                failed_rules.append(f"Age must be up to {int(max_age)}")
        else:
            if age_ok:
                score += 20
                matched_rules.append(f"Age between {int(min_age)} and {int(max_age)}")
            else:
                failed_rules.append(f"Age not in range {int(min_age)}-{int(max_age)}")

        # Income (25)
        max_income = row.get("max_income", float("inf"))
        if pd.isna(max_income):
            max_income = float("inf")
        if income <= float(max_income):
            score += 25
            if math.isinf(float(max_income)):
                matched_rules.append("No explicit income ceiling")
            else:
                matched_rules.append(f"Income <= {int(max_income)}")
        else:
            failed_rules.append(f"Income above limit ({int(max_income)})")

        # State (25)
        row_states = row.get("eligible_states", [])
        if isinstance(row_states, str):
            row_states_l = row_states.lower()
            state_ok = ("all" in row_states_l) or (state and state in row_states_l)
        else:
            row_states_set = {str(x).lower() for x in row_states}
            state_ok = ("all" in row_states_set) or (state in row_states_set)

        if state_ok or not state:
            score += 25
            matched_rules.append("State matched")
        else:
            failed_rules.append("State not matched")

        # Category (20)
        row_categories = row.get("eligible_categories", [])
        if isinstance(row_categories, str):
            cat_ok = (category in row_categories.lower()) if category else True
        else:
            cat_ok = (not row_categories) or (category in {str(x).lower() for x in row_categories}) if category else True

        if cat_ok:
            score += 20
            matched_rules.append("Category matched or open")
        else:
            failed_rules.append("Category not matched")

        # Occupation/status (10)
        row_occ = row.get("occupation_tags", [])
        if isinstance(row_occ, str):
            occ_ok = (occupation in row_occ.lower()) if occupation else True
        else:
            occ_ok = (not row_occ) or (occupation in {str(x).lower() for x in row_occ}) if occupation else True

        if occ_ok:
            score += 10
            matched_rules.append("Occupation/status matched or open")
        else:
            failed_rules.append("Occupation/status not matched")

        if score >= threshold:
            results.append({
                "scheme_name": row.get(scheme_col, "Unknown Scheme"),
                "match_score": int(score),
                "why_eligible": "; ".join(matched_rules[:3]),
                "matched_rules": matched_rules,
                "failed_rules": failed_rules,
                "benefits": row.get(benefits_col, "") if benefits_col else "",
                "documents": row.get(documents_col, "") if documents_col else "",
                "application_link": row.get(application_col, "") if application_col else "",
                "eligibility_text": row.get("eligibility_raw", "")
            })

    results = sorted(results, key=lambda x: x["match_score"], reverse=True)
    return results[:top_k]


# Quick validation sample
sample_user = {
    "age": 28,
    "state": "Maharashtra",
    "income": 45000,
    "category": "SC",
    "occupation": "student"
}

sample_results = get_eligible_schemes(sample_user, top_k=5)
print(f"Top matches found: {len(sample_results)}")
pd.DataFrame(sample_results)[["scheme_name", "match_score", "why_eligible"]].head(5)

Processed dataset saved: C:/Users/ASUS/OneDrive/Documents/GITHUB/SchemeSathi/Dataset/schemes_processed.csv
Top matches found: 5


,scheme_name,match_score,why_eligible
0,Karma Veer Dadasaheb Gaikwad Sabalikaran & Swa...,100,Age between 18 and 60; No explicit income ceil...
1,Mahila Kisan Yojana (Maharashtra),100,Age between 18 and 50; Income <= 120000; State...
2,Micro Credit Finance,100,Age between 18 and 50; Income <= 120000; State...
3,Agriculture Academic Studies Scheme,88,No specific age criteria mentioned; No explici...
4,Assam Arogya Nidhi Scheme,88,No specific age criteria mentioned; No explici...


## Testing with Multiple Profiles
We test the matcher with different user personas to verify relevance and stability.

In [67]:
test_profiles = [
    {"name": "Student_SC_MH", "age": 20, "state": "Maharashtra", "income": 30000, "category": "SC", "occupation": "student"},
    {"name": "Farmer_OBC_UP", "age": 42, "state": "Uttar Pradesh", "income": 60000, "category": "OBC", "occupation": "farmer"},
    {"name": "Senior_General_KA", "age": 67, "state": "Karnataka", "income": 25000, "category": "General", "occupation": "senior citizen"},
    {"name": "Woman_Entrepreneur_TN", "age": 31, "state": "Tamil Nadu", "income": 90000, "category": "General", "occupation": "entrepreneur"},
    {"name": "Worker_ST_OD", "age": 36, "state": "Odisha", "income": 40000, "category": "ST", "occupation": "worker"},
]

for p in test_profiles:
    results = get_eligible_schemes(p, top_k=5)
    print(f"\n=== {p['name']} | Matches: {len(results)} ===")
    if results:
        display(pd.DataFrame(results)[["scheme_name", "match_score", "why_eligible"]].head(5))
    else:
        print("No matches found.")


=== Student_SC_MH | Matches: 5 ===


,scheme_name,match_score,why_eligible
0,Financial Assistance To Cultural Organizations...,100,Age up to 25; No explicit income ceiling; Stat...
1,Karma Veer Dadasaheb Gaikwad Sabalikaran & Swa...,100,Age between 18 and 60; No explicit income ceil...
2,Mahila Kisan Yojana (Maharashtra),100,Age between 18 and 50; Income <= 120000; State...
3,Micro Credit Finance,100,Age between 18 and 50; Income <= 120000; State...
4,Shaktidoot Scheme,100,Age up to 20; No explicit income ceiling; Stat...



=== Farmer_OBC_UP | Matches: 5 ===


,scheme_name,match_score,why_eligible
0,Centrally Sponsored Scheme of Post-Matric Scho...,88,No specific age criteria mentioned; No explici...
1,Computer Training Scheme,88,No specific age criteria mentioned; Income <= ...
2,ICAR Junior Research Fellowship For Post-Gradu...,88,No specific age criteria mentioned; No explici...
3,ICAR Senior Research Fellowship For Post-Gradu...,88,No specific age criteria mentioned; No explici...
4,Agriculture Academic Studies Scheme,78,No specific age criteria mentioned; No explici...



=== Senior_General_KA | Matches: 5 ===


,scheme_name,match_score,why_eligible
0,ICAR Junior Research Fellowship For Post-Gradu...,88,No specific age criteria mentioned; No explici...
1,Airavata Scheme,80,Age 21+; Income <= 500000; State matched
2,Prerana (micro Credit Finance) Scheme,80,Age 21+; Income <= 200000; State matched
3,Shaktidoot Scheme,80,No explicit income ceiling; State matched; Cat...
4,Agriculture Academic Studies Scheme,78,No specific age criteria mentioned; No explici...



=== Woman_Entrepreneur_TN | Matches: 5 ===


,scheme_name,match_score,why_eligible
0,New Entrepreneur-cum-enterprise Development Sc...,100,Age between 21 and 35; No explicit income ceil...
1,Afforestation Schemes Providing Incentives And...,88,No specific age criteria mentioned; No explici...
2,Chief Minister’s Comprehensive Health Insuranc...,88,No specific age criteria mentioned; Income <= ...
3,District Central Cooperative Banks and through...,88,No specific age criteria mentioned; No explici...
4,District Central Cooperative Banks and through...,88,No specific age criteria mentioned; No explici...



=== Worker_ST_OD | Matches: 5 ===


,scheme_name,match_score,why_eligible
0,"""Input Assistance to Farmers for Taking up Fis...",100,Age 5+; No explicit income ceiling; State matched
1,Marriage Assistance (O.B.O.C.W.W.B),100,Age 18+; No explicit income ceiling; State mat...
2,SUBHADRA,100,Age up to 60; Income <= 250000; State matched
3,Assam Arogya Nidhi Scheme,88,No specific age criteria mentioned; No explici...
4,Assistance for Funeral Expenses (O.B.O.C.W.W.B),88,No specific age criteria mentioned; No explici...


## Output Schema Check
This check confirms that the function returns all fields needed by the frontend team.

In [69]:
sample = get_eligible_schemes(
    {"age": 28, "state": "Maharashtra", "income": 45000, "category": "SC", "occupation": "student"},
    top_k=1
)

required_keys = {
    "scheme_name", "match_score", "why_eligible", "matched_rules", "failed_rules",
    "benefits", "documents", "application_link", "eligibility_text"
}

if not sample:
    print("No sample result found. Lower threshold and test again.")
else:
    missing = required_keys - set(sample[0].keys())
    print("Missing keys:", missing if missing else "None")
    print("Schema OK" if not missing else "Schema not complete")
    display(pd.DataFrame(sample))

Missing keys: None
Schema OK


,scheme_name,match_score,why_eligible,matched_rules,failed_rules,benefits,documents,application_link,eligibility_text
0,Karma Veer Dadasaheb Gaikwad Sabalikaran & Swa...,100,Age between 18 and 60; No explicit income ceil...,"[Age between 18 and 60, No explicit income cei...",[],The beneficiary is provided with 2-acre irriga...,BPL Card Aadhaar Card. Proof of Age (Birth Cer...,Step 1: Visit the District Social Welfare Offi...,The applicant should be a citizen of India. Th...


### SchemeSathi Matching API Contract

Input profile (dict):
- age: int
- state: str
- income: float
- category: str
- occupation: str

Function:
- get_eligible_schemes(user_profile, top_k=10, threshold=55)

Output (list of dict):
- scheme_name
- match_score
- why_eligible
- matched_rules
- failed_rules
- benefits
- documents
- application_link
- eligibility_text

In [77]:
# Save sample outputs for frontend/demo integration
demo_profiles = [
    {"age": 20, "state": "Maharashtra", "income": 30000, "category": "SC", "occupation": "student"},
    {"age": 42, "state": "Uttar Pradesh", "income": 60000, "category": "OBC", "occupation": "farmer"},
    {"age": 67, "state": "Karnataka", "income": 25000, "category": "General", "occupation": "senior citizen"}
]

demo_outputs = {f"profile_{i+1}": get_eligible_schemes(p, top_k=5) for i, p in enumerate(demo_profiles)}
import json
with open("demo_outputs.json", "w", encoding="utf-8") as f:
    json.dump(demo_outputs, f, indent=2, ensure_ascii=False)
print("Saved demo_outputs.json")

Saved demo_outputs.json


In [ ]:
# Optional NLP section run order:
# 1) Build TF-IDF index
# 2) Run NLP search function
# 3) Run hybrid recommendation function
print("NLP setup note: Run the next 3 NLP cells from top to bottom")

In [73]:
# Hybrid: eligibility filter + NLP ranking

def get_hybrid_recommendations(user_profile, user_query="", top_k=8):
    eligible = get_eligible_schemes(user_profile, top_k=100, threshold=55)
    if not eligible:
        return []

    if not str(user_query).strip():
        return eligible[:top_k]

    q_vec = vectorizer.transform([str(user_query)])

    reranked = []
    for item in eligible:
        sname = item.get("scheme_name", "")
        matches = df_nlp[df_nlp[scheme_col].astype(str) == str(sname)] if scheme_col in df_nlp.columns else pd.DataFrame()

        if matches.empty:
            nlp_score = 0.0
        else:
            idx = matches.index[0]
            nlp_score = float(cosine_similarity(q_vec, X_tfidf[idx]).flatten()[0]) * 100

        final_score = round((0.7 * item["match_score"]) + (0.3 * nlp_score), 2)
        merged = dict(item)
        merged["nlp_relevance"] = round(nlp_score, 2)
        merged["final_score"] = final_score
        reranked.append(merged)

    reranked = sorted(reranked, key=lambda x: x["final_score"], reverse=True)
    return reranked[:top_k]


hybrid_demo = get_hybrid_recommendations(
    user_profile={"age": 20, "state": "Maharashtra", "income": 30000, "category": "SC", "occupation": "student"},
    user_query="education scholarship and student hostel support",
    top_k=5
)

pd.DataFrame(hybrid_demo)[["scheme_name", "match_score", "nlp_relevance", "final_score", "why_eligible"]] if hybrid_demo else "No matches"

,scheme_name,match_score,nlp_relevance,final_score,why_eligible
0,Shaktidoot Scheme,100,2.85,70.86,Age up to 20; No explicit income ceiling; Stat...
1,Micro Credit Finance,100,2.04,70.61,Age between 18 and 50; Income <= 120000; State...
2,E-Medhabruti,88,28.11,70.03,No specific age criteria mentioned; Income <= ...
3,Financial Assistance To Cultural Organizations...,100,0.00,70.00,Age up to 25; No explicit income ceiling; Stat...
4,Karma Veer Dadasaheb Gaikwad Sabalikaran & Swa...,100,0.00,70.00,Age between 18 and 60; No explicit income ceil...


In [72]:
# NLP-only query retrieval (classic yojanagpt style)
def find_schemes_by_query(query, top_k=5):
    q = str(query).strip()
    if not q:
        return []

    q_vec = vectorizer.transform([q])
    sims = cosine_similarity(q_vec, X_tfidf).flatten()
    top_idx = sims.argsort()[::-1][:top_k]

    out = []
    for i in top_idx:
        if sims[i] <= 0:
            continue
        row = df_nlp.iloc[i]
        out.append({
            "scheme_name": row.get(scheme_col, "Unknown Scheme"),
            "nlp_relevance": round(float(sims[i]) * 100, 2),
            "benefits": row.get(benefits_col, "") if benefits_col else "",
            "eligibility_text": row.get("eligibility_raw", "")
        })
    return out


find_schemes_by_query("scholarship for SC students in Maharashtra", top_k=5)

[{'scheme_name': 'State Merit Scholarship Manipur',
  'nlp_relevance': 36.19,
  'benefits': 'For Class X passed students: The scholarship provides Rs. 6,000 per annum to the first 300 students who passed the High School Leaving Certificate Examination (HSLC) conducted by BSEM for a period of two years. For Class XII passed students: The scholarship provides Rs. 12,000 per annum to the first 125 students of Science, 125 students of Social Sciences, and 50 students of Commerce who passed the class XII examination conducted by COHSEM for a period of three years. The scholarship is available to students who continue their studies within or outside the state but within the country, starting from the academic session of 2015-16. The scholarship aims to provide equal opportunities for higher education to deserving students, regardless of their economic background. In addition to the initial list of recipients, 50 students will be kept on the waiting list for possible refusal, ineligibility, o

In [71]:
# Build searchable text corpus for NLP retrieval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def _safe_text(val):
    return "" if pd.isna(val) else str(val)


scheme_text_cols = [
    c for c in [scheme_col, eligibility_col, benefits_col, documents_col, application_col] if c is not None
]

df_nlp = df_phase.copy()
df_nlp["nlp_text"] = df_nlp.apply(
    lambda r: " ".join(_safe_text(r.get(c, "")) for c in scheme_text_cols),
    axis=1
)

vectorizer = TfidfVectorizer(stop_words="english", max_features=6000, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(df_nlp["nlp_text"])

In [80]:
# Quality and evaluation utilities

def run_data_quality_checks(data):
    report = {}
    report["total_rows"] = int(len(data))

    # duplicated() needs hashable values, so convert list/dict cells to string form.
    safe_df = data.copy()
    for col in safe_df.columns:
        safe_df[col] = safe_df[col].apply(lambda x: str(x) if isinstance(x, (list, dict, set)) else x)

    report["duplicate_rows"] = int(safe_df.duplicated().sum())

    key_cols = [c for c in [scheme_col, eligibility_col, benefits_col, documents_col, application_col] if c is not None]
    missing_map = {}
    for c in key_cols:
        missing_map[c] = int(data[c].isna().sum())
    report["missing_by_key_columns"] = missing_map

    # Remove exact duplicate rows using the hash-safe view
    keep_idx = ~safe_df.duplicated()
    cleaned = data.loc[keep_idx].copy()
    report["rows_after_dedup"] = int(len(cleaned))
    return report, cleaned


def build_nlp_index(data):
    # Local imports keep this cell self-contained
    from sklearn.feature_extraction.text import TfidfVectorizer

    def _safe_text(v):
        return "" if pd.isna(v) else str(v)

    text_cols = [c for c in [scheme_col, eligibility_col, benefits_col, documents_col, application_col] if c is not None]
    local_df = data.copy()
    local_df["nlp_text"] = local_df.apply(
        lambda r: " ".join(_safe_text(r.get(c, "")) for c in text_cols),
        axis=1,
    )

    vec = TfidfVectorizer(stop_words="english", max_features=7000, ngram_range=(1, 2))
    mat = vec.fit_transform(local_df["nlp_text"])
    return local_df, vec, mat


def _normalize_tokens(text):
    raw = re.findall(r"[a-zA-Z]+", str(text).lower())
    stop = {"for", "and", "the", "with", "in", "of", "to", "a", "an", "on", "support", "scheme"}
    return [t for t in raw if t not in stop and len(t) > 2]


def _keyword_overlap_boost(query, candidate_text):
    q_tokens = set(_normalize_tokens(query))
    c_tokens = set(_normalize_tokens(candidate_text))
    if not q_tokens:
        return 0.0
    overlap = q_tokens.intersection(c_tokens)
    # Scale overlap to a 0-100-like score space for blending with tf-idf similarity.
    return min(100.0, (len(overlap) / len(q_tokens)) * 100.0)


def find_schemes_by_query_robust(query, local_df, vec, mat, top_k=5):
    from sklearn.metrics.pairwise import cosine_similarity

    q = str(query).strip()
    if not q:
        return []

    q_vec = vec.transform([q])
    sims = cosine_similarity(q_vec, mat).flatten()
    top_idx = sims.argsort()[::-1][: max(top_k * 4, 20)]

    out = []
    for i in top_idx:
        if sims[i] <= 0:
            continue
        row = local_df.iloc[i]
        candidate_text = f"{row.get(scheme_col, '')} {row.get('nlp_text', '')}"
        overlap_boost = _keyword_overlap_boost(q, candidate_text)
        final_nlp = (0.75 * float(sims[i]) * 100.0) + (0.25 * overlap_boost)

        out.append({
            "scheme_name": row.get(scheme_col, "Unknown Scheme"),
            "nlp_relevance": round(final_nlp, 2),
            "benefits": row.get(benefits_col, "") if benefits_col else "",
            "eligibility_text": row.get("eligibility_raw", ""),
        })

    out = sorted(out, key=lambda x: x["nlp_relevance"], reverse=True)
    return out[:top_k]


def get_hybrid_recommendations_robust(user_profile, user_query, local_df, vec, mat, top_k=8):
    from sklearn.metrics.pairwise import cosine_similarity

    eligible = get_eligible_schemes(user_profile, top_k=100, threshold=55)
    if not eligible:
        return []

    if not str(user_query).strip():
        return eligible[:top_k]

    q = str(user_query)
    q_vec = vec.transform([q])

    merged_results = []
    for item in eligible:
        sname = str(item.get("scheme_name", ""))
        matches = local_df[local_df[scheme_col].astype(str) == sname] if scheme_col in local_df.columns else pd.DataFrame()

        if matches.empty:
            tfidf_score = 0.0
            overlap_boost = 0.0
        else:
            idx = matches.index[0]
            tfidf_score = float(cosine_similarity(q_vec, mat[idx]).flatten()[0]) * 100
            candidate_text = f"{sname} {matches.iloc[0].get('nlp_text', '')}"
            overlap_boost = _keyword_overlap_boost(q, candidate_text)

        nlp_score = (0.75 * tfidf_score) + (0.25 * overlap_boost)
        final_score = round((0.6 * item["match_score"]) + (0.4 * nlp_score), 2)

        m = dict(item)
        m["nlp_relevance"] = round(nlp_score, 2)
        m["final_score"] = final_score
        merged_results.append(m)

    merged_results = sorted(merged_results, key=lambda x: x["final_score"], reverse=True)
    return merged_results[:top_k]


def evaluate_engine(test_cases, local_df, vec, mat):
    # A lightweight offline score for judging readiness
    scores = []
    for tc in test_cases:
        res = get_hybrid_recommendations_robust(tc["profile"], tc["query"], local_df, vec, mat, top_k=5)
        top_names = [str(x["scheme_name"]).lower() for x in res]
        hit = any(k.lower() in " ".join(top_names) for k in tc.get("expected_keywords", []))
        scores.append(1 if hit else 0)

    accuracy = (sum(scores) / len(scores)) * 100 if scores else 0
    return round(accuracy, 2), scores


# Run hardening pipeline
quality_report, df_phase_dedup = run_data_quality_checks(df_phase)
df_nlp_v2, vectorizer_v2, X_tfidf_v2 = build_nlp_index(df_phase_dedup)

benchmark_cases = [
    {
        "profile": {"age": 20, "state": "Maharashtra", "income": 30000, "category": "SC", "occupation": "student"},
        "query": "scholarship for students",
        "expected_keywords": ["scholarship", "education", "student"],
    },
    {
        "profile": {"age": 42, "state": "Uttar Pradesh", "income": 60000, "category": "OBC", "occupation": "farmer"},
        "query": "farmer support and agriculture",
        "expected_keywords": ["farmer", "agri", "kisan"],
    },
    {
        "profile": {"age": 67, "state": "Karnataka", "income": 25000, "category": "General", "occupation": "senior citizen"},
        "query": "old age pension and senior citizen support",
        "expected_keywords": ["senior", "pension", "old age"],
    },
]

eval_accuracy, eval_vector = evaluate_engine(benchmark_cases, df_nlp_v2, vectorizer_v2, X_tfidf_v2)

print("Data quality report:", quality_report)
print("Benchmark accuracy (% hit in top-5):", eval_accuracy)
print("Case-wise hits:", eval_vector)

# Demo robust hybrid output
demo_hybrid_v2 = get_hybrid_recommendations_robust(
    user_profile={"age": 20, "state": "Maharashtra", "income": 30000, "category": "SC", "occupation": "student"},
    user_query="education scholarship and student hostel support",
    local_df=df_nlp_v2,
    vec=vectorizer_v2,
    mat=X_tfidf_v2,
    top_k=5,
)

pd.DataFrame(demo_hybrid_v2)[["scheme_name", "match_score", "nlp_relevance", "final_score", "why_eligible"]] if demo_hybrid_v2 else "No matches"

Data quality report: {'total_rows': 3400, 'duplicate_rows': 3, 'missing_by_key_columns': {'scheme_name': 0, 'eligibility': 0, 'benefits': 0, 'documents': 0, 'application': 0}, 'rows_after_dedup': 3397}
Benchmark accuracy (% hit in top-5): 66.67
Case-wise hits: [1, 1, 0]


,scheme_name,match_score,nlp_relevance,final_score,why_eligible
0,E-Medhabruti,88,37.65,67.86,No specific age criteria mentioned; Income <= ...
1,State Technical Scholarship for ST Student,88,28.96,64.38,No specific age criteria mentioned; No explici...
2,Savitribai Phule Scholarship For V.J.N.T. And ...,88,27.29,63.72,No specific age criteria mentioned; No explici...
3,Technical Education Scholarship Scheme For Deg...,88,24.43,62.57,No specific age criteria mentioned; Income <= ...
4,State Pre-metric Scholarship For Disabled,88,22.14,61.65,No specific age criteria mentioned; No explici...


## Final Hardening (Recommended)
This section improves project quality for hackathon judging.

What this adds:
1. Data quality checks (duplicates, missing fields)
2. Robust NLP index builder (no fragile run-order dependency)
3. Hybrid recommender wrapper
4. Simple evaluation score for your matching engine

In [84]:
# Final recommender with explanation and confidence for UI

def get_final_recommendations(user_profile, user_query, top_k=5):
    # 1) Eligibility candidates
    eligible = get_eligible_schemes(user_profile, top_k=120, threshold=55)
    if not eligible:
        return []

    # 2) NLP score map
    q_norm = normalize_mixed_query(user_query)
    nlp_rank = find_schemes_by_query_robust(q_norm, df_nlp_v2, vectorizer_v2, X_tfidf_v2, top_k=60)
    nlp_map = {str(x["scheme_name"]): float(x["nlp_relevance"]) for x in nlp_rank}

    # 3) Semantic score map
    sem_rank = semantic_search_schemes(q_norm, semantic_df, semantic_embeddings, top_k=60)
    sem_map = {str(x["scheme_name"]): float(x["semantic_score"]) for x in sem_rank}

    # 4) Blend using adaptive weights
    fused = []
    for e in eligible:
        name = str(e["scheme_name"])
        nlp_s = nlp_map.get(name, 0.0)
        sem_s = sem_map.get(name, 0.0)
        final_score = (
            adaptive_weights["eligibility"] * float(e["match_score"])
            + adaptive_weights["nlp"] * nlp_s
            + adaptive_weights["semantic"] * sem_s
        )

        item = dict(e)
        item["nlp_relevance"] = round(nlp_s, 2)
        item["semantic_score"] = round(sem_s, 2)
        item["final_score"] = round(final_score, 2)
        fused.append(item)

    fused = sorted(fused, key=lambda x: x["final_score"], reverse=True)[:top_k]

    # Confidence = top score + margin from second result
    if len(fused) == 1:
        confidence = min(99.0, fused[0]["final_score"])
        margin = fused[0]["final_score"]
    else:
        margin = fused[0]["final_score"] - fused[1]["final_score"]
        confidence = min(99.0, max(40.0, fused[0]["final_score"] + 0.4 * margin))

    # UI explanation for rank #1
    top = fused[0]
    top["ui_explanation"] = {
        "confidence": round(confidence, 2),
        "why_ranked_1": [
            f"High eligibility match: {top.get('match_score', 0)}",
            f"NLP relevance to query: {top.get('nlp_relevance', 0)}",
            f"Semantic relevance: {top.get('semantic_score', 0)}",
            f"Matched rules: {', '.join(top.get('matched_rules', [])[:3])}",
        ],
        "weights_used": dict(adaptive_weights),
        "score_margin_vs_rank2": round(margin, 2),
    }

    return fused


# Demo run
final_demo = get_final_recommendations(
    user_profile={"age": 20, "state": "Maharashtra", "income": 30000, "category": "SC", "occupation": "student"},
    user_query="Maharashtra SC chatravriti yojana for student",
    top_k=5,
)

pd.DataFrame(final_demo)[["scheme_name", "match_score", "nlp_relevance", "semantic_score", "final_score", "why_eligible"]] if final_demo else "No matches"

# Example feedback update (thumbs up on rank 1)
if final_demo:
    new_weights = apply_feedback(final_demo[0], vote=1)
    print("Updated adaptive weights after thumbs up:", new_weights)
    print("UI explanation for rank #1:", final_demo[0].get("ui_explanation", {}))

Updated adaptive weights after thumbs up: {'eligibility': 0.5631067961165049, 'nlp': 0.24271844660194175, 'semantic': 0.1941747572815534}
UI explanation for rank #1: {'confidence': 71.34, 'why_ranked_1': ['High eligibility match: 88', 'NLP relevance to query: 36.52', 'Semantic relevance: 67.78', 'Matched rules: No specific age criteria mentioned, No explicit income ceiling, State matched'], 'weights_used': {'eligibility': 0.55, 'nlp': 0.25, 'semantic': 0.2}, 'score_margin_vs_rank2': 0.62}


In [85]:
# Updated benchmark using final recommender (eligibility + nlp + semantic)

def evaluate_final_recommender(test_cases, top_k=5):
    hits = []
    for tc in test_cases:
        out = get_final_recommendations(tc["profile"], tc["query"], top_k=top_k)
        names = " ".join([str(x.get("scheme_name", "")).lower() for x in out])
        hit = any(k.lower() in names for k in tc.get("expected_keywords", []))
        hits.append(1 if hit else 0)

    score = (sum(hits) / len(hits)) * 100 if hits else 0.0
    return round(score, 2), hits


final_benchmark_cases = [
    {
        "profile": {"age": 20, "state": "Maharashtra", "income": 30000, "category": "SC", "occupation": "student"},
        "query": "SC chatravriti yojana for students",
        "expected_keywords": ["scholarship", "student", "education"],
    },
    {
        "profile": {"age": 42, "state": "Uttar Pradesh", "income": 60000, "category": "OBC", "occupation": "farmer"},
        "query": "kisan krishi support scheme",
        "expected_keywords": ["farmer", "agri", "kisan"],
    },
    {
        "profile": {"age": 67, "state": "Karnataka", "income": 25000, "category": "General", "occupation": "senior citizen"},
        "query": "buzurg pension sahayata",
        "expected_keywords": ["pension", "senior", "old age"],
    },
    {
        "profile": {"age": 30, "state": "Tamil Nadu", "income": 90000, "category": "General", "occupation": "entrepreneur"},
        "query": "business loan women entrepreneur support",
        "expected_keywords": ["entrepreneur", "loan", "self employment"],
    },
    {
        "profile": {"age": 35, "state": "Odisha", "income": 40000, "category": "ST", "occupation": "worker"},
        "query": "divyang ya mazdoor welfare",
        "expected_keywords": ["worker", "labour", "welfare"],
    },
]

final_accuracy, final_hits = evaluate_final_recommender(final_benchmark_cases, top_k=5)
print("Final benchmark accuracy (% hit in top-5):", final_accuracy)
print("Case-wise hits:", final_hits)
print("Current adaptive weights:", adaptive_weights)

Final benchmark accuracy (% hit in top-5): 80.0
Case-wise hits: [1, 1, 1, 1, 0]
Current adaptive weights: {'eligibility': 0.5631067961165049, 'nlp': 0.24271844660194175, 'semantic': 0.1941747572815534}


In [83]:
# Sentence-transformers semantic search + adaptive feedback weighting
from sentence_transformers import SentenceTransformer, util

# A compact model suitable for hackathons (fast + decent semantic quality)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def build_embedding_index(data):
    text_cols = [c for c in [scheme_col, eligibility_col, benefits_col, documents_col, application_col] if c is not None]

    local_df = data.copy()
    local_df["semantic_text"] = local_df.apply(
        lambda r: " ".join(str(r.get(c, "")) for c in text_cols), axis=1
    )

    emb = embed_model.encode(local_df["semantic_text"].tolist(), convert_to_tensor=True, show_progress_bar=False)
    return local_df, emb


def semantic_search_schemes(query, local_df, embeddings, top_k=5):
    q_norm = normalize_mixed_query(query)
    q_emb = embed_model.encode(q_norm, convert_to_tensor=True, show_progress_bar=False)
    sims = util.cos_sim(q_emb, embeddings)[0].cpu().numpy()

    top_idx = sims.argsort()[::-1][:top_k]
    out = []
    for idx in top_idx:
        row = local_df.iloc[idx]
        out.append({
            "scheme_name": row.get(scheme_col, "Unknown Scheme"),
            "semantic_score": round(float(sims[idx]) * 100, 2),
            "benefits": row.get(benefits_col, "") if benefits_col else "",
            "eligibility_text": row.get("eligibility_raw", ""),
        })
    return out


# Adaptive ranking weights updated from user feedback (thumbs up/down)
adaptive_weights = {"eligibility": 0.55, "nlp": 0.25, "semantic": 0.20}
feedback_log = []


def _normalize_weights(w):
    s = sum(w.values())
    for k in w:
        w[k] = max(0.1, min(0.8, w[k] / s))
    # Normalize again after clipping
    s2 = sum(w.values())
    for k in w:
        w[k] = w[k] / s2
    return w


def apply_feedback(selected_result, vote):
    # vote: +1 thumbs up, -1 thumbs down
    global adaptive_weights
    lr = 0.03

    elig = float(selected_result.get("match_score", 0))
    nlp = float(selected_result.get("nlp_relevance", 0))
    sem = float(selected_result.get("semantic_score", 0))

    # Reward the strongest contributing signal when user likes result
    contrib = {"eligibility": elig, "nlp": nlp, "semantic": sem}
    top_signal = max(contrib, key=contrib.get)

    if vote > 0:
        adaptive_weights[top_signal] += lr
    else:
        adaptive_weights[top_signal] -= lr

    adaptive_weights = _normalize_weights(adaptive_weights)
    feedback_log.append({"vote": vote, "top_signal": top_signal, "weights": dict(adaptive_weights)})
    return dict(adaptive_weights)


# Build semantic index once
semantic_df, semantic_embeddings = build_embedding_index(df_phase_dedup)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [82]:
# Multilingual normalization for Hindi-English (Hinglish) queries
import re

HINGLISH_MAP = {
    "yojana": "scheme",
    "yojna": "scheme",
    "chatravriti": "scholarship",
    "scholarship": "scholarship",
    "kisan": "farmer",
    "krishi": "agriculture",
    "buzurg": "senior citizen",
    "vriddh": "senior citizen",
    "pension": "pension",
    "mahila": "women",
    "vidhyarthi": "student",
    "vidyarthi": "student",
    "rojgar": "employment",
    "berozgar": "unemployed",
    "viklang": "disabled",
    "divyang": "disabled",
    "awas": "housing",
    "shiksha": "education",
    "swasthya": "health",
    "aay": "income",
    "garib": "low income",
}


def normalize_mixed_query(query):
    q = str(query).lower().strip()
    q = re.sub(r"[^a-z0-9\s]", " ", q)
    tokens = [t for t in q.split() if t]
    mapped = [HINGLISH_MAP.get(t, t) for t in tokens]
    return " ".join(mapped)


print(normalize_mixed_query("SC chatravriti yojana Maharashtra student"))

sc scholarship scheme maharashtra student


In [81]:
# Install semantic retrieval model library (run once)
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## Advanced Upgrades (ML + NLP + Product Readiness)
This section adds stronger semantic retrieval, adaptive feedback learning, mixed-language query normalization, and UI-ready confidence explanations.